In [0]:
# Enable Change Data Feed for vector search synchronization
spark.sql(f"ALTER TABLE dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

# Display sample data to understand table structure
display(spark.sql(f"SELECT * FROM dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked LIMIT 5"))
     

path,chunk,id
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf,"Spark Adaptive Query Execution (AQE) Internals Prepared By: Kaviraj T U How Spark Rewrites Your Plan at Runtime Many engineers tune Spark jobs manually: Adjust shuffle partitions Change join strategy Handle skew manually Increase executor memory But modern Spark can do something powerful: It can change the execution plan at runtime. That feature is called Adaptive Query Execution (AQE). And understanding its internals is a senior-level Spark skill. What is AQE? Normally, Spark: Builds a logical plan 2 Optimizes it Creates a physical plan Executes it That plan is fixed before execution starts. Spark Adaptive Query Execution (AQE) Internals == page == Problem? Spark doesn't know the real/data distribution until shuffle happens. AQE solves this. It collects runtime statistics and modifies the physical plan dynamically. When Does AQE Kick In? AQE activates after a shuffle stage completes. At that moment Spark now knows: Actual partition sizes Real data distribution True join sizes Skewed partitions Using this information, it can re-optimize the next stage. What AQE Actually Optimizes Dynamic Shuffle Partition Coalescing Default shuffle partitions: spark.sql.shuffle.partitions=200 But what if data is small? Without AQE: → 200 tiny partitions Spark Adaptive Query Execution (AQE) Internals 2 == page == Too many small tasks Scheduling overhead With AQE: → Spark merges small partitions automatically Fewer, optimal-sized tasks Result: Less overhead Better resource usage 2 Dynamic Join Strategy Switching At planning time, Spark may choose: Sort-Merge Join. But at runtime it may discover: One side is actually small enough to broadcast. AQE can switch: Sort-Merge → Broadcast Join Without you changing code. This can reduce shuffle dramatically. Skew Join Optimization Spark Adaptive Query Execution (AQE) Internals 3",0
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf,"== page == If one partition is much larger: Without AQE: One executor overloaded → Long-running task Potential OOM With AQE: → Spark splits skewed partition → Processes it in parallel → Reduces imbalance This is huge for production workloads. How to Enable AQE spark.conf.set(""spark.sql.adaptive.enabled"", ""true"") Other useful configs: spark.conf.set(""spark.sql.adaptive.coalescePartitions.enabled"", ""true"") spark.conf.set(""spark.sql.adaptive.skewJoin.enabled"", ""true"") In Spark 3+, AQE is often enabled by default. Spark Adaptive Query Execution (AQE) Internals == page == How to Verify AQE is Working In Spark UI: Look at SQL tab Check ""AdaptiveSparkPlan"" in physical plan Compare initial plan vs final plan You'll see plan changes during execution. That's AQE in action. When AQE Doesn't Help AQE is powerful — but not magic. It won't fix: Poor partitioning strategy Massive data skew beyond threshold Bad UDF design Memory misconfiguration It optimizes around runtime statistics — not architectural flaws. Production Insight Spark Adaptive Query Execution (AQE) Internals 5 == page == Before AQE: Engineers manually tuned everything. After AQE: - Spark handles many optimizations automatically. But...Senior engineers still: Inspect execution plans Monitor skew behavior Validate join strategy Tune thresholds carefully Automation helps. Understanding still wins. Final Takeaway AQE makes Spark adaptive. But if you don't understand: How shuffle works How joins are selected How partition sizing affects execution You won't know whether AQE helped — or hid a deeper issue. Distributed systems are not about writing queries. They're about understanding execution. Spark Adaptive Query Execution (AQE) Internals 6 == page == AQE is Spark's way of becoming smarter at runtime. Your job is to be smarter than the runtime. Thank You.! Happy Learning... Spark Adaptive Query Execution (AQE) Internals",1
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_gena

In [0]:
import mlflow.deployments

# Initialize deployment client for accessing embedding models
deploy_client = mlflow.deployments.get_deploy_client("databricks")

# Generate embeddings for a sample question
question = "How Generative AI impacts humans?"
response = deploy_client.predict(endpoint="databricks-gte-large-en", inputs={"input": [question]})
embeddings = [e["embedding"] for e in response.data]

# Display embedding information
print("Embedding for question:", embeddings[0])
print("Embedding shape:", len(embeddings[0]))

Embedding for question: [1.201171875, -0.01910400390625, -0.436279296875, 0.1551513671875, -0.249267578125, -0.5107421875, 0.95849609375, -1.4072265625, -1.638671875, 0.1722412109375, -0.238037109375, -0.1663818359375, 0.69921875, -0.2437744140625, -0.6455078125, -0.36083984375, 0.88134765625, -0.69189453125, 0.485107421875, -0.97802734375, -0.12457275390625, -1.1875, 0.420166015625, -0.69970703125, 1.263671875, -0.373779296875, -0.2369384765625, -0.5947265625, 0.49609375, -0.179443359375, 1.1171875, 0.423095703125, -0.72509765625, 0.5185546875, 0.282470703125, 0.1900634765625, 0.7900390625, -0.2354736328125, -0.4130859375, 0.78857421875, 0.5390625, 0.058258056640625, -0.218505859375, -0.49951171875, -0.48828125, -0.254150390625, 0.5107421875, -1.076171875, -0.57666015625, 0.3505859375, -0.2403564453125, -0.60546875, 0.48095703125, -0.211181640625, -0.41357421875, -0.4296875, 0.6162109375, -0.50048828125, -0.5244140625, 0.33447265625, -0.1644287109375, -0.15771484375, 0.369140625, -0.1

In [0]:
%pip install databricks-sdk
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.vector_search.client import VectorSearchClient

def check_vector_search_endpoint(endpoint_name: str):
    vsc = VectorSearchClient(disable_notice=True)
    try:
        endpoint = vsc.get_endpoint(endpoint_name)
    except NotFound as e:
        raise RuntimeError(f"⛔️ ERROR: Vector search endpoint '{endpoint_name}' does not exist. Details: {e}")
    status = endpoint.get("endpoint_status", {}).get("state", "")
    if status != "ONLINE":
        raise RuntimeError(f"⛔️ ERROR: Vector search endpoint '{endpoint_name}' exists but is not ready.")
    
    print("✅ Vector Search endpoint is ready. ")
    print(f"\n✍️ Vector Search endpoint to use: {endpoint_name}")

vector_search_endpoint = "rag-gen-ai-endpoint"
check_vector_search_endpoint(vector_search_endpoint)

✅ Vector Search endpoint is ready. 

✍️ Vector Search endpoint to use: rag-gen-ai-endpoint


In [0]:
from databricks.vector_search.client import VectorSearchClient

# Initialize the Vector Search client
vsc = VectorSearchClient(disable_notice=True)

# Define index name using three-level naming convention
index_name = "dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked_index"
endpoint_name = "rag-gen-ai-endpoint"
docs_table = "dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked"

# Create the index using managed embeddings with Delta Sync
vsc.create_delta_sync_index_and_wait(
    endpoint_name=endpoint_name,
    index_name=index_name,
    source_table_name=docs_table,
    primary_key='id',
    embedding_source_column="chunk",
    embedding_model_endpoint_name="databricks-gte-large-en",
    pipeline_type="TRIGGERED",
)
print(f"Index '{index_name}' created for table '{docs_table}' using endpoint '{endpoint_name}'.")

Index 'dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked_index' created for table 'dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked' using endpoint 'rag-gen-ai-endpoint'.


In [0]:
# Get the vector search index for performing searches
index = vsc.get_index(index_name=index_name)
print(index.describe())

{'name': 'dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked_index', 'endpoint_name': 'rag-gen-ai-endpoint', 'primary_key': 'id', 'index_type': 'DELTA_SYNC', 'delta_sync_index_spec': {'source_table': 'dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked', 'embedding_source_columns': [{'name': 'chunk', 'embedding_model_endpoint_name': 'databricks-gte-large-en'}], 'pipeline_type': 'TRIGGERED', 'pipeline_id': 'cf81915d-9fc4-474d-83a9-ca6946aae356'}, 'status': {'detailed_state': 'ONLINE_NO_PENDING_UPDATE', 'message': 'Index creation succeeded. Check latest status: https://adb-1969779519092963.3.azuredatabricks.net/explore/data/dmi_genomics_qa_team_consenting_practice/bg_genai/docs_chunked_index', 'indexed_row_count': 285, 'triggered_update_status': {'last_processed_commit_version': 2, 'last_processed_commit_timestamp': '2026-03-10T06:41:59Z'}, 'ready': True, 'index_url': 'adb-1969779519092963.3.azuredatabricks.net/api/2.0/vector-search/indexes/dmi_genomics_qa_team_con

In [0]:
query_text = "what is time travel in databricks?"
results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=3
)
display(results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 3,
  'data_array': [['dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/Databricks Data Engineer.pdf',
    "== page ==\n\nScalable Metadata Works with petabytes of data\nTransaction Log Internals\n\nStored in JSON files in _delta_log/\nCaptures all operations: INSERT, DELETE, UPDATE\nEach new transaction = new log file\nDelta Table Types\n\nManaged Table\n\nStored inside dbfs:/user/hive/warehouse/\nDropping the table deletes both data and metadata\nExternal Table\n\n- Uses LOCATION keyword to store data outside default path\n- Dropping only deletes metadata, data remains\nDelta Features with Detailed Use Cases\n\nTime Travel\n\nTime Travel enables you to query older snapshots of a Delta table. This is useful for debugging, audits, recovering from accidental deletes or updates.\nSyntax:\n\nSELECT * FROM my_table VERSION AS OF 3;\nSE

In [0]:
query_text = "what is time travel in databricks"
results_hybrid = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    query_type="hybrid",
    num_results=5
)
display(results_hybrid)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 5,
  'data_array': [['dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/Databricks Data Engineer.pdf',
    "== page ==\n\nScalable Metadata Works with petabytes of data\nTransaction Log Internals\n\nStored in JSON files in _delta_log/\nCaptures all operations: INSERT, DELETE, UPDATE\nEach new transaction = new log file\nDelta Table Types\n\nManaged Table\n\nStored inside dbfs:/user/hive/warehouse/\nDropping the table deletes both data and metadata\nExternal Table\n\n- Uses LOCATION keyword to store data outside default path\n- Dropping only deletes metadata, data remains\nDelta Features with Detailed Use Cases\n\nTime Travel\n\nTime Travel enables you to query older snapshots of a Delta table. This is useful for debugging, audits, recovering from accidental deletes or updates.\nSyntax:\n\nSELECT * FROM my_table VERSION AS OF 3;\nSE

In [0]:
query_text = "what is time travel in databricks?"
filtered_results = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    filters={"path LIKE" : "/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/Databricks Data Engineer.pdf"}, # TODO: Change the path if you used a different catalog and schema
    num_results=3
)

display(filtered_results)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 3,
  'data_array': [['dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/Databricks Data Engineer.pdf',
    "== page ==\n\nScalable Metadata Works with petabytes of data\nTransaction Log Internals\n\nStored in JSON files in _delta_log/\nCaptures all operations: INSERT, DELETE, UPDATE\nEach new transaction = new log file\nDelta Table Types\n\nManaged Table\n\nStored inside dbfs:/user/hive/warehouse/\nDropping the table deletes both data and metadata\nExternal Table\n\n- Uses LOCATION keyword to store data outside default path\n- Dropping only deletes metadata, data remains\nDelta Features with Detailed Use Cases\n\nTime Travel\n\nTime Travel enables you to query older snapshots of a Delta table. This is useful for debugging, audits, recovering from accidental deletes or updates.\nSyntax:\n\nSELECT * FROM my_table VERSION AS OF 3;\nSE

In [0]:
# Example: Re-rank top N results from a semantic search using DatabricksReranker

from databricks.vector_search.reranker import DatabricksReranker

query_text = "what is time travel in databricks?"
results_reranked = index.similarity_search(
    query_text=query_text,
    columns=["path", "chunk"],
    num_results=5,
    reranker=DatabricksReranker(columns_to_rerank=["chunk"])
)

display(results_reranked)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


{'manifest': {'column_count': 3,
  'columns': [{'name': 'path'}, {'name': 'chunk'}, {'name': 'score'}]},
 'result': {'row_count': 5,
  'data_array': [['dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/Databricks_complete_guide.pdf',
    "== page ==\n\nDelta Lake Operations\n\nCRUD and More\nBasic CRUD Operations\n\n# INSERT\ndf.write.format('delta').mode('append').saveAsTable('my_table')\n# UPDATE (SQL)\n# UPDATE my_table SET status = 'active' WHERE id = 1\n# DELETE (SQL)\n# DELETE FROM my_table WHERE created_at < '2023-01-01'\n# UPSERT using MERGE (most important pattern!)\n# See next section\nMERGE - The Most Important Pattern\n\nMERGE handles insert, update, and delete in one atomic operation. Essential for CDC (Change Data Capture) and SCD (Slowly Changing Dimensions).\nMERGE INTO target_table AS target\nUSING source_table AS source\nON target.id = source.id\n\nWHEN MATCHED AND source.is_deleted = true THEN\nDELETE\n\nWHEN MATCHED THEN\nUPDATE SET *\n\nW